# 15. PCA / TICA Trajectory Dimensionality Reduction

Reduce MD trajectories to a low-dimensional conformational space via PCA or TICA,
enabling noise-free clustering and slow-mode visualisation.

In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname = "LIG",
    topology_glob  = "equilibrating_topology.pdb",
    trajectory_glob = "trajectory.xtc",
)

REPLICA_ROOTS = [
    Path("../run01"),
    Path("../run02"),
]

# PCA settings
PCA_N_COMPONENTS = 3
PCA_SELECTION     = "name CA"       # Cα backbone

# TICA settings
TICA_LAG         = 10               # lag in frames
TICA_N_COMPONENTS = 3
TICA_SELECTION   = "name CA"
# ============================================================

In [ ]:
from mdatools.io.loaders import discover_replicas
from mdatools.universe import load_and_align

replicas = discover_replicas(REPLICA_ROOTS, cfg)
print(f"Found {len(replicas)} replicas: {[r['name'] for r in replicas]}")

## PCA

In [ ]:
from mdatools.analysis.dimensionality import TrajectoryPCA

pca = TrajectoryPCA(cfg, n_components=PCA_N_COMPONENTS, selection=PCA_SELECTION)

pca_results = {}
for rep in replicas:
    u = load_and_align(rep["topology"], rep["trajectory"], cfg)
    pca_results[rep["name"]] = pca.fit_transform(u, rep["name"])
    print(f"{rep['name']}: projection shape = {pca_results[rep['name']].projection.shape}")

In [ ]:
import matplotlib.pyplot as plt
from mdatools.plotting import plot_pca_landscape, plot_explained_variance, plot_pca_projection_time

for name, result in pca_results.items():
    fig = plot_pca_landscape(result)
    plt.show()

    fig = plot_explained_variance(result)
    plt.show()

    fig = plot_pca_projection_time(result)
    plt.show()

## TICA

In [ ]:
from mdatools.analysis.dimensionality import TrajectoryTICA

tica = TrajectoryTICA(cfg, lag=TICA_LAG, n_components=TICA_N_COMPONENTS, selection=TICA_SELECTION)

tica_results = {}
for rep in replicas:
    u = load_and_align(rep["topology"], rep["trajectory"], cfg)
    tica_results[rep["name"]] = tica.fit_transform(u, rep["name"])
    print(f"{rep['name']}: projection shape = {tica_results[rep['name']].projection.shape}")

In [ ]:
for name, result in tica_results.items():
    fig = plot_pca_landscape(result)
    plt.title(f"TICA landscape — {name}")
    plt.show()

    fig = plot_pca_projection_time(result)
    plt.show()

## PCA pre-processing for clustering

Pass the PCA projection to `PoseClusterer` for improved clustering.

In [ ]:
# Example: use PCA projection coordinates for Ward clustering
import numpy as np
from mdatools.analysis.clustering import PoseClusterer

rep = replicas[0]
u = load_and_align(rep["topology"], rep["trajectory"], cfg)
pca_result = pca_results[rep["name"]]

# Standard clustering on raw ligand coordinates
clusterer = PoseClusterer(cfg, method="ward", rmsd_cutoff=2.0)
cluster_result = clusterer.run(u, rep["name"])

# Plot PCA landscape coloured by cluster assignment
n_frames = len(u.trajectory)
labels = cluster_result.labels[:n_frames]  # align lengths
fig = plot_pca_landscape(pca_result, color_by=labels)
plt.title("PCA coloured by cluster")
plt.show()